<a id="top"></a>
# BB84 detector study

**Question.** At a matched 5% false-alarm target, when do basis- and
state-conditioned error scores detect a partial intercept-resend attack better
than pooled QBER, and how sensitive is that comparison to finite baseline
fitting and three interim looks?

This is a controlled Monte Carlo detector comparison, not a QKD security proof.
It runs locally and has no Qiskit, IBM Quantum, network, or QFT dependency.

## Run this notebook

1. Use a Python kernel with `numpy`, `pandas`, `scipy`, and `matplotlib`
   installed (or install the repository requirements).
2. Run the cells from top to bottom. The quick control is small; the terminal
   and cumulative grids are the longer seeded calculations.
3. Change study settings only in the two configuration cells. Rerun all cells
   below a changed setting so fitted baselines, cutoffs, and evaluation rows
   remain consistent.
4. Use `is_attack`, not `eve_p_z`, to identify attack-free rows. Null rows keep
   `eve_p_z = 0.5` only as a neutral numeric placeholder.

The principal data frames are `bb84_sweep_results` (one terminal look),
`bb84_calibration_summary` (variation across fitted baselines), and
`bb84_cumulative_results` (any alarm across 25%, 50%, and 100% looks).

## Navigate

- [Method and code map](#method-map)
- [Quick single-setting control](#quick-control)
- [Terminal detector sweeps](#terminal-sweeps)
- [Terminal results](#terminal-results)
- [Calibration-fit summary](#calibration-summary)
- [Cumulative alarms](#cumulative-alarms)
- [References](#references)


<a id="method-map"></a>
## Method and code map

| Code | What it implements | Source boundary |
| --- | --- | --- |
| `sample_bb84_checks` | Random BB84 state counts and the partial intercept-resend error channel | Four-state preparation, basis choice, sifting, and public testing follow Bennett and Brassard [R1, pp. 175–177]. For uniform Eve basis choice, the simulated attack contribution reduces to sifted QBER `f/4`, consistent with Scarani et al. [R2, §I.B.5] and Lee et al. [R3, §III.A]. The biased-Eve XOR model is a project extension. |
| `detector_scores` | Pooled QBER, basis-grouped `U2`, and state-conditioned `U4` | Separate basis error estimates are motivated by Lo, Chau, and Ardehali [R4, §4.2]. The one-sided `U2`/`U4` score weights and their comparison are project-defined statistics, not security-proof quantities. |
| `calibrate_bb84_cutoffs` | Independent null calibration with randomized handling of discrete ties | Finite-statistical treatment is motivated by Mannalath et al. [R6, Props. 1–3]. The exact Monte Carlo calibration rule is project-specific. |
| `detector_normal_moments` and `normal_alarm_probability` | Unconditional large-sample comparison including random state composition | Bernoulli QBER means, variances, and normal-tail detection calculations parallel Lee et al. [R3, Eqs. (1)–(12)]; the grouped/state score moments are project extensions. |
| `wilson_95_interval` | Evaluation uncertainty for each simulated alarm rate | Wilson's score interval [R7]. It is conditional on the fitted baseline and cutoff; between-fit variation is summarized separately. |
| `run_bb84_cumulative_sweep` | Maximum-over-looks calibration at 25%, 50%, and 100% of the check budget | The repeated-look schedule and familywise calibration are project-specific; [R6] motivates explicit finite-statistical control. |

The attack-free noise profiles are sensitivity settings. The Erven reference
uses the reported basis error rates `(q_Z, q_X) = (0.002, 0.038)` from
Erven et al. [R5, §4]; the balanced and ±90%-disparity profiles are designed
experiment factors rather than empirical device models.


<a id="quick-control"></a>
## 1. Quick single-setting control

This first pass checks the sampling and detector pipeline at one retained-check
budget. It fits four state baselines on one independent noise-only sample, sets
cutoffs on a second sample, and then reports held-out false alarms and power at
interception fraction `f = 0.10`.

The output is a smoke test, not the cross-setting result: Monte Carlo rates will
vary around their targets, and a single fitted baseline does not expose
calibration sensitivity. Use the terminal and cumulative sections for the full
comparison.


### Detector definitions and controls

| Detector | Statistic | Interpretation |
| --- | --- | --- |
| Pooled QBER | Total errors divided by total retained checks | Ignores which basis or prepared state produced an error. |
| Basis `U2` | One-sided score after grouping the two states in each basis | Can use a difference between fitted Z- and X-basis error rates. |
| State `U4` | One-sided score using all four fitted state error rates | Can use within-basis state differences; it intentionally equals `U2` when paired fitted rates are exactly equal. |

All three detectors receive the same sampled checks in each trial. Baseline
fitting, cutoff calibration, null evaluation, and attack evaluation use
independent seeded random streams. `TARGET_FALSE_ALARM` is a calibration target,
so verify the held-out null rows before comparing power.

The initial constants set the quick control and settings shared by later cells.
The terminal-sweep configuration cell sets calibration sizes, repeat fits,
simulation sizes, attack fractions, and noise profiles.


In [ ]:
import numpy as np

STATE_ORDER = ("Z0", "Z1", "X+", "X-")
DETECTOR_NAMES = ("Pooled QBER", "Basis U2", "State U4")
Q_Z, Q_X = 0.002, 0.038
STATE_BASELINE = np.repeat([Q_Z, Q_X], 2)
BASELINE_TRIALS_PER_STATE = 1_000
RETAINED_CHECKS = 50
STUDY_TRIALS = 10_000
TARGET_FALSE_ALARM = 0.05

# Settings shared by the terminal and cumulative BB84 sweeps.
EVE_Z_PROBABILITIES = (0.1, 0.5, 0.9)
CHECK_BUDGETS = (20, 50, 100)
LOOK_FRACTIONS = (0.25, 0.5, 1.0)


In [ ]:
def sample_bb84_checks(
    rng, intercept_fraction, eve_p_z, *,
    retained_checks=RETAINED_CHECKS, trials=STUDY_TRIALS, state_baseline=STATE_BASELINE
):
    """Sample state totals and errors for fixed-size sifted BB84 trials."""
    counts = rng.multinomial(retained_checks, [0.25] * len(STATE_ORDER), size=trials)
    attack_error = 0.5 * intercept_fraction * np.repeat([1 - eve_p_z, eve_p_z], 2)
    error_rate = state_baseline + (1 - 2 * state_baseline) * attack_error
    return counts, rng.binomial(counts, error_rate)


def conditioned_score(counts, errors, baseline):
    variable = (baseline > 0) & (baseline < 1)
    rates = baseline[variable]
    weights = (1 - 2 * rates) / (rates * (1 - rates))
    score = ((errors[:, variable] - counts[:, variable] * rates) * weights).sum(axis=1)

    # A true rate of 0 or 1 makes any deviation conclusive evidence against the null.
    expected = counts[:, ~variable] * baseline[~variable]
    score[(errors[:, ~variable] != expected).any(axis=1)] = np.inf
    return score


def detector_scores(counts, errors, fitted_baseline):
    """Return pooled QBER, basis-grouped U2, and state-conditioned U4."""
    basis_baseline = fitted_baseline.reshape(2, 2).mean(axis=1)
    basis_counts = counts.reshape(-1, 2, 2).sum(axis=2)
    basis_errors = errors.reshape(-1, 2, 2).sum(axis=2)
    basis_score = conditioned_score(basis_counts, basis_errors, basis_baseline)
    if np.array_equal(fitted_baseline[::2], fitted_baseline[1::2]):
        # With equal rates within each basis, the two scores are algebraically identical.
        state_score = basis_score
    else:
        state_score = conditioned_score(counts, errors, fitted_baseline)
    return np.column_stack([errors.sum(axis=1) / counts.sum(axis=1), basis_score, state_score])

In [ ]:
fit_rng, cutoff_rng, null_rng, attack_rng = [
    np.random.default_rng(seed) for seed in np.random.SeedSequence(20261030).spawn(4)
]
calibration_errors = fit_rng.binomial(BASELINE_TRIALS_PER_STATE, STATE_BASELINE)
fitted_baseline = (calibration_errors + 0.5) / (BASELINE_TRIALS_PER_STATE + 1)
cutoff_checks = sample_bb84_checks(cutoff_rng, 0.0, 0.5)
cutoff_scores = detector_scores(*cutoff_checks, fitted_baseline)
cutoffs = np.quantile(cutoff_scores, 1 - TARGET_FALSE_ALARM, axis=0, method="higher")

null_checks = sample_bb84_checks(null_rng, 0.0, 0.5)
attack_checks = sample_bb84_checks(attack_rng, 0.1, 0.5)
# A strict comparison handles ties conservatively at the cutoff.
false_alarm_rates = (detector_scores(*null_checks, fitted_baseline) > cutoffs).mean(axis=0)
detection_power = (detector_scores(*attack_checks, fitted_baseline) > cutoffs).mean(axis=0)

print("Detector       False alarm    Power at f = 0.10")
for name, false_alarm, power in zip(
    DETECTOR_NAMES, false_alarm_rates, detection_power
):
    print(f"{name:<14} {false_alarm:>10.2%} {power:>19.2%}")

<a id="terminal-sweeps"></a>
## 2. Terminal detector sweeps

This section varies the retained-check budget, Eve's interception fraction and
basis bias, the attack-free error profile, and the amount of baseline fitting
data. Each trial has one terminal decision.

For every fitted baseline and budget, an independent attack-free batch sets each
detector's cutoff. A score above the cutoff alarms; a score tied at the cutoff
alarms with the calibrated probability needed to spend the remaining 5% null
budget. Held-out batches then estimate false alarms and attack power. The same
checks are scored by all detectors and baseline fits.

The noise profiles have equal rates within a basis. Consequently, `U2` and `U4`
coincide for the true-rate control, while finite fitted samples can introduce
within-basis differences that let their scores diverge. Wilson intervals quantify
evaluation uncertainty conditional on a fit and cutoff. Repeated fits expose the
additional baseline-estimation sensitivity.

Dashed curves use unconditional normal moments, including random state
composition. Endpoint null rates make conditioned score weights singular, so
those analytic values are intentionally unavailable.

**What this yields:** `bb84_sweep_results`, with one row per noise profile,
budget, fitted baseline, attack setting, and detector.


In [ ]:
from itertools import product

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm

SWEEP_SEED = 20261031
SWEEP_CALIBRATION_SIZES = (1_000, 10_000)
SWEEP_BASELINE_FITS = 4  # Independent rate estimates per calibration size.
SWEEP_CUTOFF_TRIALS = 5_000
SWEEP_EVALUATION_TRIALS = 5_000
SWEEP_INTERCEPT_FRACTIONS = (0.005, 0.01, 0.025, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.0)

SWEEP_NOISE_PROFILES = [
    ("No noise", 0.0, 0.0),
    ("Erven reference", 0.002, 0.038),
]
for mean in (0.01, 0.02, 0.04):
    low, high = mean * (1 - 0.9), mean * (1 + 0.9)
    SWEEP_NOISE_PROFILES.extend([
        (f"{mean:.0%} mean, balanced", mean, mean),
        (f"{mean:.0%} mean, higher X", low, high),
        (f"{mean:.0%} mean, higher Z", high, low),
    ])

In [ ]:
def calibrate_bb84_cutoffs(null_scores, target):
    """Return each upper-tail cutoff and its probability of alarming on a tie."""
    rank = int(np.ceil((1 - target) * len(null_scores))) - 1
    cutoffs = np.partition(null_scores, rank, axis=0)[rank]
    above = (null_scores > cutoffs).mean(axis=0)
    tied = (null_scores == cutoffs).mean(axis=0)
    return cutoffs, np.clip((target - above) / tied, 0.0, 1.0)


def wilson_95_interval(events, trials):
    rate = events / trials
    z = 1.959963984540054
    denominator = 1 + z**2 / trials
    center = (rate + z**2 / (2 * trials)) / denominator
    radius = z * np.sqrt(rate * (1 - rate) / trials + z**2 / (4 * trials**2)) / denominator
    return np.clip(center - radius, 0.0, 1.0), np.clip(center + radius, 0.0, 1.0)


def detector_normal_moments(retained_checks, true_baseline, intercept_fraction, eve_p_z, fitted_baseline):
    """Normal moments under multinomially sampled prepared states."""
    attack_error = 0.5 * intercept_fraction * np.repeat([1 - eve_p_z, eve_p_z], 2)
    alternative = true_baseline + (1 - 2 * true_baseline) * attack_error
    pooled_mean = alternative.mean()
    means = [pooled_mean]
    variances = [pooled_mean * (1 - pooled_mean) / retained_checks]

    basis_fit = fitted_baseline.reshape(2, 2).mean(axis=1).repeat(2)
    for detector_fit in (basis_fit, fitted_baseline):
        if np.any((detector_fit <= 0) | (detector_fit >= 1)):
            means.append(np.nan)
            variances.append(np.nan)
            continue
        weights = (1 - 2 * detector_fit) / (detector_fit * (1 - detector_fit))
        state_means = weights * (alternative - detector_fit)
        per_bit_mean = state_means.mean()
        per_bit_variance = np.mean(
            weights**2 * alternative * (1 - alternative)
            + (state_means - per_bit_mean) ** 2
        )
        means.append(retained_checks * per_bit_mean)
        variances.append(retained_checks * per_bit_variance)
    return np.asarray(means), np.sqrt(np.asarray(variances))


def normal_alarm_probability(
    cutoffs, tie_rates, means, standard_deviations, retained_checks
):
    probabilities = np.empty(len(cutoffs))
    for i, (cutoff, tie_rate, mean, deviation) in enumerate(
        zip(cutoffs, tie_rates, means, standard_deviations)
    ):
        if np.isnan(deviation):
            probabilities[i] = np.nan
        elif deviation == 0:
            probabilities[i] = float(mean > cutoff) + tie_rate * float(mean == cutoff)
        elif i == 0:
            cutoff_count = cutoff * retained_checks
            mean_count = mean * retained_checks
            deviation_count = deviation * retained_checks
            above = norm.sf(
                (cutoff_count + 0.5 - mean_count) / deviation_count
            )
            tied = norm.cdf(
                (cutoff_count + 0.5 - mean_count) / deviation_count
            ) - norm.cdf(
                (cutoff_count - 0.5 - mean_count) / deviation_count
            )
            probabilities[i] = above + tie_rate * tied
        else:
            probabilities[i] = norm.sf((cutoff - mean) / deviation)
    return probabilities

In [ ]:
def run_bb84_sweep():
    fit_rng, cutoff_rng, null_rng, attack_rng = [
        np.random.default_rng(seed) for seed in np.random.SeedSequence(SWEEP_SEED).spawn(4)
    ]
    # Use a neutral Eve-basis reference for attack-free rows to keep the column numeric.
    scenarios = [(0.0, 0.5, False)]
    scenarios.extend((fraction, eve_p_z, True) for fraction, eve_p_z in product(
        SWEEP_INTERCEPT_FRACTIONS, EVE_Z_PROBABILITIES
    ))
    rows = []
    for noise, q_z, q_x in SWEEP_NOISE_PROFILES:
        true_rates = np.repeat([q_z, q_x], 2)
        fits = [(0, 1, true_rates)]
        for size in SWEEP_CALIBRATION_SIZES:
            errors = fit_rng.binomial(size, true_rates, size=(SWEEP_BASELINE_FITS, len(STATE_ORDER)))
            rates = (errors + 0.5) / (size + 1)
            fits.extend((size, index, fit) for index, fit in enumerate(rates, start=1))

        for budget in CHECK_BUDGETS:
            sampling = dict(retained_checks=budget, state_baseline=true_rates)
            cutoff_checks = sample_bb84_checks(
                cutoff_rng, 0.0, 0.5, trials=SWEEP_CUTOFF_TRIALS, **sampling
            )
            rules = [
                calibrate_bb84_cutoffs(detector_scores(*cutoff_checks, fit), TARGET_FALSE_ALARM)
                for _, _, fit in fits
            ]

            for fraction, eve_p_z, is_attack in scenarios:
                rng = attack_rng if is_attack else null_rng
                checks = sample_bb84_checks(
                    rng, fraction, eve_p_z,
                    trials=SWEEP_EVALUATION_TRIALS, **sampling
                )
                tie_draws = rng.random((SWEEP_EVALUATION_TRIALS, 1))
                for (size, fit_id, fit), (cutoffs, tie_rates) in zip(fits, rules):
                    scores = detector_scores(*checks, fit)
                    alarms = (scores > cutoffs) | ((scores == cutoffs) & (tie_draws < tie_rates))
                    event_counts = alarms.sum(axis=0)
                    lower, upper = wilson_95_interval(event_counts, SWEEP_EVALUATION_TRIALS)
                    normal_means, normal_deviations = detector_normal_moments(
                        budget, true_rates, fraction, eve_p_z, fit
                    )
                    normal_rates = normal_alarm_probability(
                        cutoffs, tie_rates, normal_means, normal_deviations, budget
                    )

                    for i, detector in enumerate(DETECTOR_NAMES):
                        rows.append({
                            "noise_profile": noise,
                            "q_z": q_z,
                            "q_x": q_x,
                            "retained_checks": budget,
                            "calibration_trials_per_state": size,
                            "baseline_fit": fit_id,
                            "qhat_z0": fit[0],
                            "qhat_z1": fit[1],
                            "qhat_x_plus": fit[2],
                            "qhat_x_minus": fit[3],
                            "intercept_fraction": fraction,
                            "eve_p_z": eve_p_z,
                            "is_attack": is_attack,
                            "detector": detector,
                            "alarm_count": event_counts[i],
                            "trials": SWEEP_EVALUATION_TRIALS,
                            "alarm_rate": event_counts[i] / SWEEP_EVALUATION_TRIALS,
                            "normal_approximation": normal_rates[i],
                            "lower_95": lower[i],
                            "upper_95": upper[i],
                            "cutoff": cutoffs[i],
                            "cutoff_tie_probability": tie_rates[i],
                        })
    return pd.DataFrame(rows)


bb84_sweep_results = run_bb84_sweep()
bb84_sweep_results.head()

<a id="terminal-results"></a>
### Interpret terminal results

Each row of `bb84_sweep_results` describes one detector and baseline fit at one
setting. Rows with `is_attack = False` give realized held-out false-alarm rates;
positive interception fractions give power.

Choose one noise profile, calibration size, fit, and Eve basis probability in
the view cell. Solid curves are Monte Carlo estimates, dashed curves are the
normal comparison, and bands are 95% Wilson intervals. The summary's 80% column
is the first **tested** interception fraction reaching estimated power of at
least 80%; it is not an interpolated detection boundary and remains blank when
no tested fraction qualifies.


### BB84 result map

| Output | Use |
| --- | --- |
| `alarm_rate` | Held-out false-alarm rate for null rows and power for attack rows |
| `normal_approximation` | Large-sample comparison for the terminal detector |
| `lower_95`, `upper_95` | Wilson interval for the simulated alarm rate |
| `first_tested_fraction_at_80pct_power` | First sampled interception fraction reaching 80% power |
| `bb84_calibration_summary` | Variation across independent fitted baselines |


In [ ]:
VIEW_NOISE_PROFILE = "2% mean, higher X"
VIEW_CALIBRATION_SIZE = 1_000
VIEW_BASELINE_FIT = 1
VIEW_EVE_P_Z = 0.5

view = bb84_sweep_results[
    bb84_sweep_results["noise_profile"].eq(VIEW_NOISE_PROFILE)
    & bb84_sweep_results["calibration_trials_per_state"].eq(VIEW_CALIBRATION_SIZE)
    & bb84_sweep_results["baseline_fit"].eq(VIEW_BASELINE_FIT)
]
if view.empty or VIEW_EVE_P_Z not in EVE_Z_PROBABILITIES:
    raise ValueError("Select a noise profile, calibration size, fit, and Eve basis from the sweep.")

power = view[view["is_attack"] & view["eve_p_z"].eq(VIEW_EVE_P_Z)]
fig, axes = plt.subplots(
    1, len(CHECK_BUDGETS), figsize=(4.6 * len(CHECK_BUDGETS), 4.2),
    sharey=True, squeeze=False, layout="constrained"
)
for ax, budget in zip(axes[0], CHECK_BUDGETS):
    budget_view = view[view["retained_checks"].eq(budget)]
    for detector in DETECTOR_NAMES:
        curve = budget_view[
            budget_view["detector"].eq(detector)
            & (~budget_view["is_attack"] | budget_view["eve_p_z"].eq(VIEW_EVE_P_Z))
        ].sort_values("intercept_fraction")
        x = curve["intercept_fraction"].to_numpy()
        line, = ax.plot(x, curve["alarm_rate"].to_numpy(), marker="o", ms=3, label=detector)
        ax.plot(
            x, curve["normal_approximation"].to_numpy(),
            color=line.get_color(), linestyle="--", linewidth=1
        )
        ax.fill_between(
            x, curve["lower_95"].to_numpy(), curve["upper_95"].to_numpy(),
            color=line.get_color(), alpha=0.15
        )
    ax.axhline(0.8, color="0.5", linestyle=":", linewidth=1)
    ax.set(title=f"{budget} retained checks", xlabel="Interception fraction", ylim=(0, 1.02))
axes[0, 0].set_ylabel("Alarm probability")
axes[0, -1].legend(loc="lower right")
fig.suptitle(
    f"{VIEW_NOISE_PROFILE} · Eve Z probability {VIEW_EVE_P_Z:g}"
    " · solid Monte Carlo, dashed normal approximation"
)
plt.show()

keys = ["retained_checks", "detector"]
false_alarms = view[~view["is_attack"]].set_index(keys)[
    ["alarm_rate", "lower_95", "upper_95"]
].rename(columns={"alarm_rate": "false_alarm_rate"})
first_at_80 = (
    power[power["alarm_rate"].ge(0.8)].groupby(keys)["intercept_fraction"].min()
    .rename("first_tested_fraction_at_80pct_power")
)
false_alarms.join(first_at_80).round(4)

<a id="calibration-summary"></a>
## 3. Calibration-fit summary

`bb84_calibration_summary` aggregates the terminal results across independently
fitted attack-free baselines. Compare `between_fit_std`, the fitwise extrema,
and the mean alarm rate before treating a detector's apparent power advantage as
stable. Four fits are enough to reveal sensitivity in this notebook, but they
are not a final estimate of the calibration-uncertainty distribution.


In [ ]:
bb84_calibration_summary = (
    bb84_sweep_results[
        bb84_sweep_results["calibration_trials_per_state"].gt(0)
    ]
    .groupby([
        "noise_profile", "retained_checks",
        "calibration_trials_per_state", "intercept_fraction",
        "eve_p_z", "is_attack", "detector",
    ], as_index=False)
    .agg(
        mean_alarm_rate=("alarm_rate", "mean"),
        between_fit_std=("alarm_rate", "std"),
        minimum_alarm_rate=("alarm_rate", "min"),
        maximum_alarm_rate=("alarm_rate", "max"),
        mean_normal_approximation=("normal_approximation", "mean"),
    )
)
bb84_calibration_summary.head()


<a id="cumulative-alarms"></a>
## 4. Cumulative three-look alarms

This section evaluates 25%, 50%, and 100% of every retained-check budget. Each
detector is standardized at each look under its fitted null. A separate
noise-only batch calibrates the maximum standardized score across looks, so the
target is a 5% probability of **any** alarm rather than 5% at each look. One
randomized tie draw is shared across looks within a trial.

**What this yields:** `bb84_cumulative_results`, including the probability of any
alarm, Wilson intervals, first-alarm timing, and reach probabilities. Check the
attack-free `any_alarm_rate` first; then compare power curves at a common budget,
noise profile, fit, and Eve basis bias. `mean_first_alarm_checks` is conditional
on an alarm occurring.

The retained-check budgets and look schedule come from the project design.
Maximum-over-looks standardization, shared tie randomization, and familywise
cutoff calibration are project-specific methods motivated by the need for
finite-statistical control in [R6].


In [ ]:
CUMULATIVE_SEED = 20261101
CUMULATIVE_CUTOFF_TRIALS = 3_000
CUMULATIVE_EVALUATION_TRIALS = 3_000


def bb84_look_counts(retained_checks):
    return np.unique(np.maximum(
        1, np.ceil(retained_checks * np.asarray(LOOK_FRACTIONS)).astype(int)
    ))


def sample_bb84_check_looks(
    rng, looks, intercept_fraction, eve_p_z, *, trials, state_baseline
):
    counts = np.zeros((trials, len(STATE_ORDER)), dtype=int)
    errors = np.zeros_like(counts)
    counts_at_looks, errors_at_looks = [], []
    previous = 0
    for look in looks:
        increment = sample_bb84_checks(
            rng, intercept_fraction, eve_p_z,
            retained_checks=int(look - previous), trials=trials,
            state_baseline=state_baseline,
        )
        counts += increment[0]
        errors += increment[1]
        counts_at_looks.append(counts.copy())
        errors_at_looks.append(errors.copy())
        previous = look
    return np.stack(counts_at_looks, axis=1), np.stack(errors_at_looks, axis=1)


def detector_scores_at_looks(counts, errors, fitted_baseline):
    return np.stack([
        detector_scores(counts[:, look], errors[:, look], fitted_baseline)
        for look in range(counts.shape[1])
    ], axis=1)


def cumulative_null_standardization(looks, fitted_baseline):
    centers, scales = [], []
    for look in looks:
        means, deviations = detector_normal_moments(
            int(look), fitted_baseline, 0.0, 0.5, fitted_baseline
        )
        centers.append([means[0], 0.0, 0.0])
        scales.append(np.nan_to_num(deviations, nan=0.0))
    return np.asarray(centers), np.asarray(scales)


def standardize_look_scores(scores, centers, scales):
    delta = scores - centers
    standardized = np.zeros_like(delta, dtype=float)
    np.divide(delta, scales, out=standardized, where=scales > 0)
    zero_scale = np.broadcast_to(scales == 0, delta.shape)
    standardized[zero_scale & (delta > 0)] = np.inf
    standardized[zero_scale & (delta < 0)] = -np.inf
    return standardized


def run_bb84_cumulative_sweep():
    fit_rng, cutoff_rng, null_rng, attack_rng = [
        np.random.default_rng(seed)
        for seed in np.random.SeedSequence(CUMULATIVE_SEED).spawn(4)
    ]
    scenarios = [(0.0, 0.5, False)]
    scenarios.extend((fraction, eve_p_z, True) for fraction, eve_p_z in product(
        SWEEP_INTERCEPT_FRACTIONS, EVE_Z_PROBABILITIES
    ))
    rows = []
    for noise, q_z, q_x in SWEEP_NOISE_PROFILES:
        true_rates = np.repeat([q_z, q_x], 2)
        fits = [(0, 1, true_rates)]
        for size in SWEEP_CALIBRATION_SIZES:
            calibration_errors = fit_rng.binomial(
                size, true_rates, size=(SWEEP_BASELINE_FITS, len(STATE_ORDER))
            )
            fitted_rates = (calibration_errors + 0.5) / (size + 1)
            fits.extend(
                (size, index, fit)
                for index, fit in enumerate(fitted_rates, start=1)
            )

        for budget in CHECK_BUDGETS:
            looks = bb84_look_counts(budget)
            cutoff_counts, cutoff_errors = sample_bb84_check_looks(
                cutoff_rng, looks, 0.0, 0.5,
                trials=CUMULATIVE_CUTOFF_TRIALS, state_baseline=true_rates,
            )
            rules = []
            for _, _, fit in fits:
                centers, scales = cumulative_null_standardization(looks, fit)
                cutoff_scores = detector_scores_at_looks(
                    cutoff_counts, cutoff_errors, fit
                )
                maxima = standardize_look_scores(
                    cutoff_scores, centers, scales
                ).max(axis=1)
                cutoffs, tie_rates = calibrate_bb84_cutoffs(
                    maxima, TARGET_FALSE_ALARM
                )
                rules.append((centers, scales, cutoffs, tie_rates))

            for fraction, eve_p_z, is_attack in scenarios:
                rng = attack_rng if is_attack else null_rng
                counts, errors = sample_bb84_check_looks(
                    rng, looks, fraction, eve_p_z,
                    trials=CUMULATIVE_EVALUATION_TRIALS,
                    state_baseline=true_rates,
                )
                tie_draws = rng.random((CUMULATIVE_EVALUATION_TRIALS, 1, 1))
                for (size, fit_id, fit), rule in zip(fits, rules):
                    centers, scales, cutoffs, tie_rates = rule
                    scores = detector_scores_at_looks(counts, errors, fit)
                    standardized = standardize_look_scores(
                        scores, centers, scales
                    )
                    alarm_at_look = (
                        (standardized > cutoffs)
                        | (
                            (standardized == cutoffs)
                            & (tie_draws < tie_rates.reshape(1, 1, -1))
                        )
                    )
                    any_alarm = alarm_at_look.any(axis=1)
                    event_counts = any_alarm.sum(axis=0)
                    lower, upper = wilson_95_interval(
                        event_counts, CUMULATIVE_EVALUATION_TRIALS
                    )
                    for detector_index, detector in enumerate(DETECTOR_NAMES):
                        detector_alarms = alarm_at_look[:, :, detector_index]
                        ever = any_alarm[:, detector_index]
                        first_indices = np.argmax(detector_alarms, axis=1)
                        first_checks = looks[first_indices]
                        row = {
                            "noise_profile": noise,
                            "q_z": q_z,
                            "q_x": q_x,
                            "retained_checks": budget,
                            "look_counts": tuple(int(value) for value in looks),
                            "calibration_trials_per_state": size,
                            "baseline_fit": fit_id,
                            "intercept_fraction": fraction,
                            "eve_p_z": eve_p_z,
                            "is_attack": is_attack,
                            "detector": detector,
                            "alarm_count": event_counts[detector_index],
                            "trials": CUMULATIVE_EVALUATION_TRIALS,
                            "any_alarm_rate": (
                                event_counts[detector_index]
                                / CUMULATIVE_EVALUATION_TRIALS
                            ),
                            "lower_95": lower[detector_index],
                            "upper_95": upper[detector_index],
                            "maximum_cutoff": cutoffs[detector_index],
                            "cutoff_tie_probability": tie_rates[detector_index],
                            "mean_first_alarm_checks": (
                                first_checks[ever].mean()
                                if ever.any() else np.nan
                            ),
                        }
                        for look_index, look in enumerate(looks):
                            label = int(round(
                                100 * LOOK_FRACTIONS[look_index]
                            ))
                            row[f"alarm_by_{label}pct"] = detector_alarms[
                                :, :look_index + 1
                            ].any(axis=1).mean()
                            row[f"first_alarm_at_{label}pct"] = (
                                ever & (first_indices == look_index)
                            ).mean()
                            row[f"reach_{label}pct"] = (
                                ~detector_alarms[
                                    :, :look_index
                                ].any(axis=1)
                            ).mean()
                        rows.append(row)
    return pd.DataFrame(rows)


bb84_cumulative_results = run_bb84_cumulative_sweep()
bb84_cumulative_results.head()


In [ ]:
cumulative_view = bb84_cumulative_results[
    bb84_cumulative_results["noise_profile"].eq(VIEW_NOISE_PROFILE)
    & bb84_cumulative_results[
        "calibration_trials_per_state"
    ].eq(VIEW_CALIBRATION_SIZE)
    & bb84_cumulative_results["baseline_fit"].eq(VIEW_BASELINE_FIT)
]
cumulative_null = cumulative_view[~cumulative_view["is_attack"]]
cumulative_power = cumulative_view[
    cumulative_view["is_attack"]
    & cumulative_view["eve_p_z"].eq(VIEW_EVE_P_Z)
]

fig, axes = plt.subplots(
    1, len(CHECK_BUDGETS),
    figsize=(4.6 * len(CHECK_BUDGETS), 4.2),
    sharey=True, squeeze=False, layout="constrained",
)
for ax, budget in zip(axes[0], CHECK_BUDGETS):
    for detector in DETECTOR_NAMES:
        null_row = cumulative_null[
            cumulative_null["retained_checks"].eq(budget)
            & cumulative_null["detector"].eq(detector)
        ]
        curve = cumulative_power[
            cumulative_power["retained_checks"].eq(budget)
            & cumulative_power["detector"].eq(detector)
        ].sort_values("intercept_fraction")
        x = np.r_[0.0, curve["intercept_fraction"].to_numpy()]
        y = np.r_[
            null_row["any_alarm_rate"].iloc[0],
            curve["any_alarm_rate"].to_numpy(),
        ]
        ax.plot(x, y, marker="o", ms=3, label=detector)
    ax.axhline(0.8, color="0.5", linestyle=":", linewidth=1)
    ax.set(
        title=f"{budget} retained checks",
        xlabel="Interception fraction", ylim=(0, 1.02),
    )
axes[0, 0].set_ylabel("Probability of any alarm")
axes[0, -1].legend(loc="lower right")
fig.suptitle(
    f"Cumulative three-look test · {VIEW_NOISE_PROFILE}"
    f" · Eve Z probability {VIEW_EVE_P_Z:g}"
)
plt.show()

cumulative_null.set_index(["retained_checks", "detector"])[
    ["look_counts", "any_alarm_rate", "lower_95", "upper_95"]
].round(4)


<a id="references"></a>
## References

- **[R1]** C. H. Bennett and G. Brassard, “Quantum Cryptography: Public Key
  Distribution and Coin Tossing,” *Proceedings of IEEE International Conference
  on Computers, Systems and Signal Processing*, pp. 175–179 (1984).
  [Reprint: arXiv:2003.06557](https://arxiv.org/abs/2003.06557).
- **[R2]** V. Scarani et al., “The Security of Practical Quantum Key
  Distribution,” *Reviews of Modern Physics* **81**, 1301–1350 (2009),
  especially §I.B.5, p. 1305.
  [arXiv:0802.4155](https://arxiv.org/abs/0802.4155).
- **[R3]** Chankyun Lee, Eunjoo Lee, and Wonhyuk Lee, “Analysis on Eavesdropper Detection in
  BB84 Quantum Key Distribution Protocol Against Partial Intercept-and-Resend
  Attack,” *ONDM 2023*, especially §III.A, Eqs. (1)–(12).
  [Open paper](https://dl.ifip.org/db/conf/ondm/ondm2023/1570879964.pdf).
- **[R4]** H.-K. Lo, H. F. Chau, and M. Ardehali, “Efficient Quantum Key
  Distribution Scheme and a Proof of Its Unconditional Security,” *Journal of
  Cryptology* **18**, 133–165 (2005), especially §4.2, pp. 26–28 of the preprint.
  [arXiv:quant-ph/0011056](https://arxiv.org/abs/quant-ph/0011056).
- **[R5]** C. Erven, X. Ma, R. Laflamme, and G. Weihs, “Entangled Quantum Key
  Distribution with a Biased Basis Choice,” *New Journal of Physics* **11**,
  045025 (2009), especially §4, p. 9.
  [arXiv:0901.0960](https://arxiv.org/abs/0901.0960).
- **[R6]** Vaisakh Mannalath, Víctor Zapatero, and Marcos Curty, “Sharp Finite Statistics for
  Quantum Key Distribution,” *Physical Review Letters* **135**, 020803 (2025),
  especially Propositions 1–3.
  [arXiv:2410.04095](https://arxiv.org/abs/2410.04095).
- **[R7]** E. B. Wilson, “Probable Inference, the Law of Succession, and
  Statistical Inference,” *Journal of the American Statistical Association*
  **22**(158), 209–212 (1927).
  [doi:10.1080/01621459.1927.10502953](https://doi.org/10.1080/01621459.1927.10502953).

[Back to top](#top)
